In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("base_path_col", "s3://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/process/ETL_Ref", "Base Path col")
dbutils.widgets.text("file_prefix", "ODM.EDW.VEN130FA.WEEKLY.ASCII.PROD", "File Prefix")
dbutils.widgets.text("column_name", "DESCRIPTION", "Column to Validate")
dbutils.widgets.text("error_report_table", "oh_apm_stg.vendor_extracts.error_load_report_log_cpc_Stg_Ref", "Error Report Table")

In [0]:

BASE = dbutils.widgets.get("base_path_col").rstrip("/")
FILE_PREFIX = dbutils.widgets.get("file_prefix")
COL_NAME = dbutils.widgets.get("column_name")
ERROR_TABLE = dbutils.widgets.get("error_report_table")

print(f"base_path_col={BASE}")
print(f"file_prefix={FILE_PREFIX}")
print(f"column_name={COL_NAME}")
print(f"error_report_table={ERROR_TABLE}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime
import re

nb_start = datetime.now()
nb_start_str = nb_start.strftime("%Y-%m-%d %H:%M:%S")

def resolve_file_by_prefix(base_dir: str, prefix: str, ext: str = ".gz"):
    try:
        entries = dbutils.fs.ls(base_dir)
    except Exception as e:
        return None, None, f"Unable to list dir {base_dir}: {e}"

    rx = re.compile(rf"^({re.escape(prefix)})\..*{re.escape(ext)}$")
    candidates = []
    for it in entries:
        if rx.match(it.name):
            candidates.append((it.path, it.name, it.modificationTime))

    if not candidates:
        return None, None, f"No file found for prefix '{prefix}' with extension {ext}"

    candidates.sort(key=lambda x: x[2], reverse=True)
    best = candidates[0]
    return best[0], best[1], None

def read_csv_gz(path: str):
    return (spark.read
                 .option("header", "true")
                 .option("delimiter", "|")
                 .csv(path))

def summarize_pipe_in_column(df, file_name: str, target_col: str,
                              date_received: str, start_str: str, end_str: str):
    
    col_actual = None
    for c in df.columns:
        if c.lower() == target_col.lower():
            col_actual = c
            break

    if col_actual is None:
        total = df.count()
        return {
            "File_Name": file_name,
            "Date_Received": date_received,
            "Start_Load_Date": start_str,
            "End_Load_Date": end_str,
            "Row_Number": int(total),
            "Error_Description": f"Column '{target_col}' not found; all {total} rows considered invalid."
        }

    pipe_df = df.where(F.col(col_actual).contains("|"))
    pipe_total = pipe_df.count()
    
    if pipe_total > 0:
        print(f"🔍 Showing first 10 problematic rows:")
        display(pipe_df.select(col_actual).limit(10))

    if pipe_total == 0:
        return None

    val_disp = F.coalesce(F.col(col_actual).cast("string"), F.lit("<NULL>"))
    counts_df = (pipe_df.groupBy(val_disp.alias("value"))
                          .agg(F.count(F.lit(1)).alias("cnt"))
                          .orderBy(F.desc("cnt"), F.asc("value")))

    parts = []
    for r in counts_df.collect():
        v, n = r["value"], int(r["cnt"])
        parts.append(f"Contains '|': '{v}' (count={n})")

    description = "; ".join(parts)

    return {
        "File_Name": file_name,
        "Date_Received": date_received,
        "Start_Load_Date": start_str,
        "End_Load_Date": end_str,
        "Row_Number": int(pipe_total),
        "Error_Description": description
    }

file_path, file_name, err = resolve_file_by_prefix(BASE, FILE_PREFIX, ".gz")
print(f"File -> {file_name or err}")

DATE_RECEIVED = nb_start_str
nb_end_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

rows_to_write = []

if file_path is None:
    rows_to_write.append({
        "File_Name": f"{FILE_PREFIX}.<date>.gz",
        "Date_Received": DATE_RECEIVED,
        "Start_Load_Date": nb_start_str,
        "End_Load_Date": nb_end_str,
        "Row_Number": 0,
        "Error_Description": "File not found by prefix."
    })
else:
    try:
        df = read_csv_gz(file_path)
        result = summarize_pipe_in_column(df, file_name, COL_NAME,
                                          DATE_RECEIVED, nb_start_str, nb_end_str)
        if result: rows_to_write.append(result)
    except Exception as e:
        rows_to_write.append({
            "File_Name": file_name,
            "Date_Received": DATE_RECEIVED,
            "Start_Load_Date": nb_start_str,
            "End_Load_Date": nb_end_str,
            "Row_Number": 0,
            "Error_Description": f"Read error: {str(e)}"
        })

if rows_to_write:
    display(spark.createDataFrame(
        rows_to_write,
        schema=T.StructType([
            T.StructField("File_Name", T.StringType()),
            T.StructField("Date_Received", T.StringType()),
            T.StructField("Start_Load_Date", T.StringType()),
            T.StructField("End_Load_Date", T.StringType()),
            T.StructField("Row_Number", T.LongType()),
            T.StructField("Error_Description", T.StringType()),
        ])
    ))
else:
    print("✅ No pipe characters found in the column.")

UPLOAD_TO_TABLE = len(rows_to_write) > 0
ROWS_PREPARED = len(rows_to_write)

dbutils.jobs.taskValues.set(key="error_load_report_flag", value=UPLOAD_TO_TABLE)
print(f"Prepared {ROWS_PREPARED} row(s). UPLOAD_TO_TABLE = {UPLOAD_TO_TABLE}")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

Col_Validation_error = False

if UPLOAD_TO_TABLE:
    error_schema = StructType([
        StructField("File_Name", StringType(), True),
        StructField("Date_Received", StringType(), True),
        StructField("Start_Load_Date", StringType(), True),
        StructField("End_Load_Date", StringType(), True),
        StructField("Row_Number", LongType(), True),
        StructField("Error_Description", StringType(), True),
    ])

    error_df = spark.createDataFrame(rows_to_write, schema=error_schema)

    if error_df.count() > 0:
        Col_Validation_error = True
        error_df.write.format("delta").mode("append").saveAsTable(ERROR_TABLE)
        print(f"❌ Validation errors found. Wrote {error_df.count()} row(s) to {ERROR_TABLE}")
    else:
        print("✅ No validation errors found. Skipping upload.")
else:
    print("⚠️ UPLOAD_TO_TABLE is False. Skipping error upload.")


print(f"Col_Validation_error = {Col_Validation_error}")
dbutils.jobs.taskValues.set(key="Col_Validation_error", value=Col_Validation_error)
